In [ ]:
import numpy as np
import pandas as pd

RUTA_DATOS = "data/Prestamos_Data_Alumnos_v5.xlsx"
PAGOS_POR_ANIO = 12   


def cargar_prestamos(ruta: str) -> pd.DataFrame:
    "Lee el dataset y deja solo las columnas necesarias, sin nulos ni duplicados."
    prestamos = pd.read_excel(ruta)
    prestamos = prestamos[["ID", "Monto_Inicial", "Ratio_Interes", "Duracion"]]
    prestamos = prestamos.dropna().drop_duplicates().reset_index(drop=True)
    prestamos["Duracion"] = prestamos["Duracion"].astype(int)
    return prestamos


def expandir_periodos(prestamos: pd.DataFrame) -> pd.DataFrame:
    "Repite cada préstamo una vez por cada mes de su duración y numera los periodos."
    cuadro = prestamos.loc[prestamos.index.repeat(prestamos["Duracion"])].copy()
    cuadro["Periodo"] = cuadro.groupby(level=0).cumcount() + 1
    return cuadro


prestamos = cargar_prestamos(RUTA_DATOS)
prestamos.head()

### Fórmulas del Sistema Francés

In [ ]:
def cuadro_frances(prestamos: pd.DataFrame) -> pd.DataFrame:
    cuadro = expandir_periodos(prestamos)

    C = cuadro["Monto_Inicial"]                          
    N = cuadro["Duracion"]                               
    n = cuadro["Periodo"]                                
    i = cuadro["Ratio_Interes"] / 100 / PAGOS_POR_ANIO   

    # Cuota fija
    cuadro["Cuota"] = C * i / (1 - (1 + i) ** -N)

    # Saldo pendiente tras n pagos
    cuadro["Saldo_Final"] = C * ((1 + i) ** N - (1 + i) ** n) / ((1 + i) ** N - 1)
    cuadro["Saldo_Inicial"] = C * ((1 + i) ** N - (1 + i) ** (n - 1)) / ((1 + i) ** N - 1)

    # Intereses sobre el saldo pendiente, el resto de la cuota amortiza capital
    cuadro["Intereses"] = cuadro["Saldo_Inicial"] * i
    cuadro["Capital_Amortizado"] = cuadro["Cuota"] - cuadro["Intereses"]

    return cuadro.drop(columns="Ratio_Interes")

In [ ]:
cuadro = cuadro_frances(prestamos)
cuadro.head()

### Comprobación: Vemos que al final de cada préstamo la cantidad a amortizar sea 0

In [ ]:
ultimas_cuotas = cuadro[cuadro["Periodo"] == cuadro["Duracion"]]
saldo_maximo = ultimas_cuotas["Saldo_Final"].abs().max()

if saldo_maximo < 0.01:
    print("Todos los préstamos quedan amortizados (saldo final = 0)")
else:
    print("Revisar: hay préstamos con saldo final distinto de 0")

### Ejemplo con el primer cliente del dataset

In [ ]:
cuadro.loc[0, ["Periodo", "Cuota", "Intereses", "Capital_Amortizado", "Saldo_Final"]].round(2)